# Run ICL-EDR

This notebook shows the minimal workflow for running the cleaned ICL-EDR reference implementation with a tiny synthetic example.

By default, it builds a dummy retrieval JSON and runs provider API calls on one synthetic target question. Set `BUILD_RETRIEVAL = False` or `RUN_API = False` if you only want to inspect the setup without making API calls.

The first run below is ICL-EDR only. A later optional section enables the matched CoT baseline and compares the two methods on the same target question.

## Example files

This clean release includes tiny synthetic files so the notebook can run without restricted thesis data:

- `example_data/dummy_target_question.csv`: one synthetic medical MCQ target question.
- `example_data/dummy_source_cases.csv`: four synthetic labelled source cases; two are clinically related to the target and two are unrelated.
- `example_data/dummy_retrieval.json`: generated by this notebook from the target and source CSVs.

Exact thesis reruns require local benchmark-derived target-question and retrieval files, which are not included here. You can point the runner to local files with `ICL_EDR_TARGET_CSV` and `ICL_EDR_RETRIEVAL_JSON`.


In [1]:
from pathlib import Path
import os
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "src" / "icl_edr_runner.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing src/icl_edr_runner.py")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from icl_edr_runner import load_env

load_env(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

missing_credentials = [key for key in ["OPENAI_API_KEY"] if not os.getenv(key)]
if missing_credentials:
    print("Missing credentials:", ", ".join(missing_credentials))
    print("Copy .env.example to .env, fill in your local keys, then restart or rerun the notebook kernel.")
else:
    print("Credentials loaded: OPENAI_API_KEY")

Project root: <repo-root>
Credentials loaded: OPENAI_API_KEY


In [2]:
target_csv = Path(os.getenv("ICL_EDR_TARGET_CSV", PROJECT_ROOT / "example_data" / "dummy_target_question.csv")).expanduser()
retrieval_json = Path(os.getenv("ICL_EDR_RETRIEVAL_JSON", PROJECT_ROOT / "example_data" / "dummy_retrieval.json")).expanduser()
source_csv = Path(os.getenv("ICL_EDR_SOURCE_CSV", PROJECT_ROOT / "example_data" / "dummy_source_cases.csv")).expanduser()

target_csv, retrieval_json, source_csv

(PosixPath('<repo-root>/example_data/dummy_target_question.csv'),
 PosixPath('<repo-root>/example_data/dummy_retrieval.json'),
 PosixPath('<repo-root>/example_data/dummy_source_cases.csv'))

## Optional: build the retrieval JSON

If a retrieval JSON is not already available, it can be built from a target-question CSV and a labelled source-case CSV. The source CSV should contain the same basic fields as the target CSV: question text/options, question key, gold answer letter, and gold answer text.

This step uses embedding API calls by default. It is enabled here so the example demonstrates the full retrieval-JSON build step.


In [3]:
# True: build example_data/dummy_retrieval.json from target/source CSVs using embeddings.
# False: reuse an existing retrieval JSON at retrieval_json.
BUILD_RETRIEVAL = True

if BUILD_RETRIEVAL:
    if missing_credentials:
        raise RuntimeError("OPENAI_API_KEY is required because BUILD_RETRIEVAL=True uses OpenAI embeddings. Copy .env.example to .env and fill the key.")
    if not target_csv.exists():
        raise FileNotFoundError(f"Target-question CSV not found: {target_csv}")
    if not source_csv.exists():
        raise FileNotFoundError(f"Source-case CSV not found: {source_csv}")

    from retrieval_builder import build_retrieval_json

    build_retrieval_json(
        target_csv_path=target_csv,
        source_csv_path=source_csv,
        output_json_path=retrieval_json,
        top_k=4,
        embedding_model=os.getenv("ICL_EDR_EMBEDDING_MODEL", "text-embedding-3-large"),
        source_pool_mode=os.getenv("ICL_EDR_SOURCE_POOL_MODE", "custom_source_pool"),
    )
    print("Wrote retrieval JSON:", retrieval_json)

Embedding texts:   0%|          | 0/1 [00:00<?, ?it/s]

Wrote retrieval JSON: <repo-root>/example_data/dummy_retrieval.json


In [4]:
missing = [path for path in [target_csv, retrieval_json] if not path.exists()]
if missing:
    print("Missing required local files:")
    for path in missing:
        print("-", path)
    INPUTS_READY = False
else:
    print("Input files found.")
    INPUTS_READY = True

Input files found.


## Configure a model setting

The example below uses the OpenAI provider branch. Replace `OPENAI_MODEL` with the model available in your account. For Hugging Face Inference Providers, use a `ModelSetting` with `provider="huggingface_inference_providers"`, set `hf_provider`, and set `HF_TOKEN`.

In [5]:
from icl_edr_runner import ICLEDRRunner, ModelSetting

setting = ModelSetting(
    name="openai_medium_example",
    provider="openai",
    model_id=os.getenv("OPENAI_MODEL", "gpt-5.4-mini"),
    temperature=1.0,
    max_tokens=8192,
    reasoning_effort=os.getenv("OPENAI_REASONING_EFFORT", "medium"),
    use_response_format=True,
    notes="Example OpenAI medium-reasoning ICL-EDR run.",
)

setting

ModelSetting(name='openai_medium_example', provider='openai', model_id='gpt-5.4-mini', temperature=1.0, max_tokens=8192, hf_provider='', reasoning_effort='medium', cot_temperature=None, icl_temperature=None, top_p=None, extra_body=None, use_response_format=True, system_prefix='', user_prefix='', notes='Example OpenAI medium-reasoning ICL-EDR run.')

In [6]:
# True: call the configured provider. False: write only the task plan/summaries from existing outputs.
RUN_API = True

if INPUTS_READY:
    if RUN_API and missing_credentials:
        raise RuntimeError("OPENAI_API_KEY is required because RUN_API=True. Copy .env.example to .env and fill the key.")
    runner = ICLEDRRunner(
        run_name="example_icl_edr_only_run",
        settings=[setting],
        project_root=PROJECT_ROOT,
        target_csv_path=target_csv,
        retrieval_json_path=retrieval_json,
        include_cot_baseline=False,  # Main example: run ICL-EDR only.
        retrieval_k=2,              # Number of nearest labelled cases shown to ICL-EDR.
        icl_solver_count=3,         # Solver ensemble size S.
        icl_ensemble_repeats=1,     # Independent ICL-EDR ensemble repeats per question.
        icl_judge_repeats=1,        # Routed-judge repeats when the solver ensemble disagrees.
        batch_solver_choices=True,  # Request S solver completions in one provider call when supported.
        run_api_calls=RUN_API,
        workers=8,                 # Parallel task workers for larger runs.
        max_tasks_this_run=20,      # Safety cap; 0 means no cap.
    )
    print("Runner created:", runner.run_name)
    print("Target rows:", len(runner.target_df))
else:
    runner = None
    print("Runner was not created because required input files are missing.")

Runner created: example_icl_edr_only_run
Target rows: 1


## Execute

With `RUN_API = True`, this cell calls the configured model provider and writes ICL-EDR-only outputs under `results/run_outputs/example_icl_edr_only_run/`. With `RUN_API = False`, it prints the task plan without provider calls.

In [7]:
if runner is not None:
    runner.run()

Project root: <repo-root>
Output dir: <repo-root>/results/run_outputs/example_icl_edr_only_run
Settings: ['openai_medium_example']
Target rows: 1 {'medqa_test': 1}
Run API calls: True
Include matched CoT baseline: False
ICL-EDR solver tasks: existing successes=1 missing_total=0 selected_this_run=0
ICL-EDR routed judge tasks: existing successes=0 missing_total=0 selected_this_run=0
Per-question completed rows: 1
              setting  method dataset_name  n  accuracy  mean_total_tokens  mean_api_calls  judge_rate  accuracy_pct
openai_medium_example ICL-EDR   medqa_test  1       1.0              869.0             1.0         0.0         100.0
openai_medium_example ICL-EDR     combined  1       1.0              869.0             1.0         0.0         100.0


## Optional: run matched CoT comparison

Set `RUN_COT_COMPARISON = True` to run the same target question with both ICL-EDR and the matched CoT baseline. This uses a separate output directory so the ICL-only example remains clean.

In [8]:
RUN_COT_COMPARISON = False

if INPUTS_READY and RUN_COT_COMPARISON:
    comparison_runner = ICLEDRRunner(
        run_name="example_icl_edr_with_cot_comparison",
        settings=[setting],
        project_root=PROJECT_ROOT,
        target_csv_path=target_csv,
        retrieval_json_path=retrieval_json,
        include_cot_baseline=True,   # Adds matched CoT rows to the same summary table.
        cot_rollouts=1,              # Number of CoT baseline completions per question.
        retrieval_k=2,
        icl_solver_count=3,
        icl_ensemble_repeats=1,
        icl_judge_repeats=1,
        batch_solver_choices=True,
        run_api_calls=RUN_API,
        workers=8,
        max_tasks_this_run=20,
    )
    comparison_runner.run()
else:
    comparison_runner = None
    print("Set RUN_COT_COMPARISON = True to run ICL-EDR plus matched CoT.")

Set RUN_COT_COMPARISON = True to run ICL-EDR plus matched CoT.
